# **Peilin Li - Section 2 - a. a) : Top-down beta estimation**

In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import yfinance as yf
from datetime import datetime

In [14]:
!pip install -q yfinance

# Step 1: Download historical price data

Historical price data for **Amazon (AMZN)** and the **S&P 500 Total Return Index (^SP500TR)** are downloaded from **Yahoo Finance** using the yfinance Python library.

**The S&P 500 Total Return Index** is selected as a proxy for the market portfolio because it includes both capital gains and reinvested dividends, providing a more comprehensive measure of market returns consistent with the CAPM framework.

**A five-year estimation period** (November 2021–October 2026) is chosen to balance sufficient observations for reliable beta estimation with the relevance of recent market conditions. Price data are downloaded from October 2021 to provide the initial prices needed to calculate November 2021 returns.

**Daily adjusted closing prices** (interval="1d", auto_adjust=True) are downloaded to allow accurate extraction of month-end prices and flexibility for potential robustness checks.

**Data source:** Yahoo Finance (accessed 7 November 2026).

In [15]:
# "AMZN" for Amaon, "^SP500TR" for S&P 500 total return
data = yf.download(
    ["AMZN", "^SP500TR"],
    start="2021-10-01",
    end="2026-11-01",
    interval="1d",
    auto_adjust=True,
    progress=False
)

print(data["Close"].head())
print(data["Close"].tail())

Ticker            AMZN     ^SP500TR
Date                               
2021-10-01  164.162994  9098.250000
2021-10-04  159.488998  8980.639648
2021-10-05  161.050003  9076.190430
2021-10-06  163.100494  9113.450195
2021-10-07  165.121506  9191.190430
Ticker            AMZN      ^SP500TR
Date                                
2026-10-26  249.089996  17226.599609
2026-10-27  253.100006  17329.470703
2026-10-28  254.600006  17377.560547
2026-10-29  243.789993  17324.679688
2026-10-30  240.350006  17011.140625


# Step 2: Check null data

In [16]:
# Extract adjusted closing prices
daily_prices = data["Close"].copy()

# Check missing values
print(daily_prices.isna().sum())

# Display rows containing missing values
print(daily_prices[daily_prices.isna().any(axis=1)])

Ticker
AMZN        0
^SP500TR    0
dtype: int64
Empty DataFrame
Columns: [AMZN, ^SP500TR]
Index: []


# Step 3: Get month end data
Monthly returns are used instead of daily returns to reduce short-term market noise while providing sufficient observations for a reliable beta estimation over the five-year period.

In [17]:
monthly_prices = daily_prices.resample("ME").last()

print(monthly_prices.isna().sum())
monthly_prices.head(10)

Ticker
AMZN        0
^SP500TR    0
dtype: int64


Ticker,AMZN,^SP500TR
Date,,
2021-10-31,168.621506,9625.019531
2021-11-30,175.353500,9558.330078
2021-12-31,166.716995,9986.700195
2022-01-31,149.573502,9469.919922
2022-02-28,153.563004,9186.370117
2022-03-31,162.997498,9527.459961
2022-04-30,124.281502,8696.650391
2022-05-31,120.209503,8712.599609
2022-06-30,106.209999,7993.430176


# Step 4: Calculate Historical Monthly Returns
use simple return: $$R_t = \frac{P_t}{P_{t-1}} - 1$$


In [18]:
# Calculate historical monthly returns
monthly_returns = monthly_prices.pct_change(fill_method=None).dropna()

# Check the results
print(monthly_returns.head())
print(monthly_returns.tail())
print(monthly_returns.shape)

Ticker          AMZN  ^SP500TR
Date                          
2021-11-30  0.039924 -0.006929
2021-12-31 -0.049252  0.044816
2022-01-31 -0.102830 -0.051747
2022-02-28  0.026673 -0.029942
2022-03-31  0.061437  0.037130
Ticker          AMZN  ^SP500TR
Date                          
2026-06-30 -0.119347 -0.009523
2026-07-31  0.139465 -0.000635
2026-08-31 -0.043486  0.027233
2026-09-30 -0.040882 -0.003457
2026-10-31 -0.035320 -0.008698
(60, 2)


# Step 5: Estimate Beta using OLS regression
$${R_{\mathrm{AMZN},t}=\alpha+\beta R_{m,t}+\varepsilon_t}$$

Run this OLS we get the $$\hat{\beta}$$

In [19]:
# Define dependent variable (Amazon returns)
Y = monthly_returns["AMZN"]

# Define independent variable (Market returns)
X = monthly_returns["^SP500TR"]

# Add a constant term for the intercept (alpha)
X = sm.add_constant(X)

# Run OLS regression
model = sm.OLS(Y, X).fit()

# Display regression results
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                   AMZN   R-squared:                       0.459
Model:                            OLS   Adj. R-squared:                  0.449
Method:                 Least Squares   F-statistic:                     49.13
Date:                Sat, 07 Nov 2026   Prob (F-statistic):           2.83e-09
Time:                        14:35:21   Log-Likelihood:                 73.036
No. Observations:                  60   AIC:                            -142.1
Df Residuals:                      58   BIC:                            -137.9
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0051      0.010     -0.531      0.5

In [20]:
# Extract estimated alpha and beta
alpha = model.params["const"]
beta = model.params["^SP500TR"]

print(f"Estimated Alpha: {alpha:.4f}")
print(f"Estimated Beta: {beta:.4f}")

Estimated Alpha: -0.0051
Estimated Beta: 1.4917


In [21]:
# check
# Calculate beta using covariance / variance
beta_cov = (
    monthly_returns["AMZN"].cov(monthly_returns["^SP500TR"])
    / monthly_returns["^SP500TR"].var()
)

print(f"OLS Beta: {beta:.4f}")
print(f"Covariance Beta: {beta_cov:.4f}")

OLS Beta: 1.4917
Covariance Beta: 1.4917


### Interpretation of the Outcome

The OLS regression is based on 60 monthly observations from November 2021 to October 2026, using Amazon's returns as the dependent variable and the S&P 500 Total Return Index as the market proxy.

**Estimated Beta (β = 1.4917):** Amazon's systematic risk is approximately 49.17% higher than that of the market portfolio proxy. A 1 percentage point increase in market returns is associated with an average increase of approximately 1.49 percentage points in Amazon's returns.

**Statistical Significance (p < 0.001):** The estimated beta is statistically significant, providing strong evidence of a positive relationship between Amazon's returns and market returns.

**R-squared (R² = 0.459):** Approximately 45.9% of the variation in Amazon's monthly returns is explained by market returns, while the remaining 54.1% is not explained by this single-factor regression.

**Intercept (α = −0.0051):** The estimated monthly intercept is approximately −0.51%, but it is not statistically significant (p = 0.598).

**95% Confidence Interval for Beta ([1.066, 1.918]):** The interval indicates uncertainty around the beta estimate, although its lower bound remains above 1.

**Conclusion:** Amazon exhibits greater sensitivity to market movements than the market portfolio proxy. The estimated historical beta of 1.4917 will be used as an input in the CAPM to estimate Amazon's cost of equity.

# Step 6: Estimate Cost of Equity using CAPM
$$r_e = r_f + \hat{\beta}(E[R_m] - r_f)$$

**Valuation Date:** 1 November 2026


The risk-free rate and equity risk premium (ERP) are obtained from **Aswath Damodaran's implied ERP estimates**, published by NYU Stern School of Business. This source is selected because it provides transparent, widely used estimates for corporate valuation.

The **Trailing 12-month, Adjusted Payout Implied ERP** is selected because it incorporates adjusted shareholder distributions, including share repurchases, rather than relying solely on cash dividends. It also provides a forward-looking estimate of the market risk premium based on current market valuations, making it more suitable for estimating the cost of equity than a historical average based on realized returns.

The estimates dated **1 November 2026** are used as the latest available inputs at the time of analysis. The selected **implied ERP** is **4.41%**, and the corresponding **U.S. Treasury rate** reported by Damodaran is **5.29%**.

The risk-free rate used in Damodaran's ERP calculation is adopted to maintain consistency between the two CAPM inputs, since the implied ERP is estimated relative to this rate. Using a different Treasury yield could introduce inconsistencies between the assumptions underlying the ERP and the CAPM calculation.

**Source:** [Damodaran, Equity Risk Premiums — NYU Stern](https://pages.stern.nyu.edu/~adamodar/New_Home_Page/home.htm) (1 November 2026).

In [23]:
# Estimated beta from OLS regression
beta = model.params["^SP500TR"]

# Damodaran's data (1 November 2026)
rf = 0.0529
erp = 0.0441

# CAPM
cost_of_equity = rf + beta * erp

print(f"Estimated Beta: {beta:.4f}")
print(f"Risk-free Rate: {rf:.2%}")
print(f"Equity Risk Premium: {erp:.2%}")
print(f"Estimated Cost of Equity: {cost_of_equity:.2%}")

Estimated Beta: 1.4917
Risk-free Rate: 5.29%
Equity Risk Premium: 4.41%
Estimated Cost of Equity: 11.87%


### Interpretation of Estimated Cost of Equity

Using the CAPM, Amazon's annualized **cost of equity** is estimated at **11.87%**. This represents the expected return required by equity investors to compensate for the systematic risk associated with investing in Amazon.

With an estimated beta of 1.4917, Amazon exhibits greater sensitivity to market movements than the market portfolio proxy. Consequently, investors require an additional **risk premium** of approximately **6.58%** above the **risk-free rate** of **5.29%**.

The estimated cost of equity represents Amazon's required return on equity financing and can serve as an input in corporate valuation and investment decisions. However, this estimate depends on the CAPM assumptions, the stability of the historical beta, and the selected market risk premium.

#Assumptions and Limitations

**1. Stability of Historical Beta**

The analysis assumes that Amazon's historical beta is representative of its future systematic risk. However, changes in business operations, financial leverage, and market conditions may cause beta to vary over time.

**2. Market Portfolio Proxy**

The S&P 500 Total Return Index is assumed to approximate the market portfolio. However, it primarily represents large-cap U.S. equities and does not capture the entire investable market, potentially introducing market proxy bias.

**3. CAPM Assumptions**

The CAPM assumes that expected returns are determined by exposure to systematic market risk. In practice, other risk factors may also affect expected returns, meaning that the single-factor CAPM may not fully capture Amazon's required return.

**4. Historical Data Selection**

The five-year estimation window and monthly return frequency are assumed to provide a reasonable estimate of beta. However, different sample periods or return frequencies may produce different estimates.

**5. Risk-free Rate and ERP**

The selected U.S. Treasury rate is assumed to approximate the risk-free rate, while Damodaran's implied ERP is assumed to reflect future market risk compensation. Both inputs are subject to estimation uncertainty, and the implied ERP depends on assumptions about future earnings and shareholder payouts.